# Importing Libraries

In [1]:
import os
import math
import random
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import scipy
import numpy as np
import pandas as pd

# Importing Dataframe

In [2]:
airbnb = pd.read_csv('../01. Data/01. Raw Data/airbnb_paris_weekends.csv')

In [3]:
airbnb.shape

(3558, 20)

# Cleaning the Dataframe

In [4]:
airbnb.head()

,Unnamed: 0,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat
0,0,536.396682,Entire home/apt,False,False,5.0,False,0,1,9.0,89.0,1,1.351201,0.212346,390.776775,19.001549,1030.738507,47.550371,2.35900,48.86800
1,1,290.101594,Private room,False,True,2.0,True,0,0,10.0,97.0,1,0.699821,0.193710,518.478270,25.211044,1218.658866,56.219575,2.35385,48.86282
2,2,445.754497,Entire home/apt,False,False,4.0,False,0,1,10.0,100.0,1,0.968982,0.294343,432.689942,21.039580,1069.894793,49.356741,2.36023,48.86375
3,3,211.343089,Private room,False,True,2.0,False,0,0,10.0,94.0,1,3.302319,0.234740,444.555284,21.616533,902.856370,41.650870,2.31714,48.87475
4,4,266.334234,Entire home/apt,False,False,2.0,True,0,0,9.0,88.0,1,1.402430,0.055052,1013.458689,49.279502,1348.063511,62.189313,2.33408,48.85384


In [5]:
# Check for null values
airbnb.isnull().sum()   

Unnamed: 0                    0
realSum                       0
room_type                     0
room_shared                   0
room_private                  0
person_capacity               0
host_is_superhost             0
multi                         0
biz                           0
cleanliness_rating            0
guest_satisfaction_overall    0
bedrooms                      0
dist                          0
metro_dist                    0
attr_index                    0
attr_index_norm               0
rest_index                    0
rest_index_norm               0
lng                           0
lat                           0
dtype: int64

## DataDict

realSum: the full price of accommodation for two people and two nights in EUR

room_type: the type of the accommodation

room_shared: dummy variable for shared rooms

room_private: dummy variable for private rooms

person_capacity: the maximum number of guests

host_is_superhost: dummy variable for superhost status

multi: dummy variable if the listing belongs to hosts with 2-4 offers

biz: dummy variable if the listing belongs to hosts with more than 4 offers

cleanliness_rating: cleanliness rating

guest_satisfaction_overall: overall rating of the listing

bedrooms: number of bedrooms (0 for studios)

dist: distance from city centre in km

metro_dist: distance from nearest metro station in km

attr_index: attraction index of the listing location

attr_index_norm: normalised attraction index (0-100)

rest_index: restaurant index of the listing location

attr_index_norm: normalised restaurant index (0-100)

lng: longitude of the listing location

lat: latitude of the listing location

city: city of the listing location

weekend: data for weekend or weekday. Now it's string variable, let's convert it to dummy variable: 1 for weekend, o for weekday

In [6]:
# dropping unecessary columns
airbnb.drop(columns=['Unnamed: 0', 'multi', 'biz', 'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index', 'rest_index_norm'], inplace=True)

In [7]:
# checking if dropping worked
airbnb.head()

,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,lng,lat
0,536.396682,Entire home/apt,False,False,5.0,False,9.0,89.0,1,1.351201,2.35900,48.86800
1,290.101594,Private room,False,True,2.0,True,10.0,97.0,1,0.699821,2.35385,48.86282
2,445.754497,Entire home/apt,False,False,4.0,False,10.0,100.0,1,0.968982,2.36023,48.86375
3,211.343089,Private room,False,True,2.0,False,10.0,94.0,1,3.302319,2.31714,48.87475
4,266.334234,Entire home/apt,False,False,2.0,True,9.0,88.0,1,1.402430,2.33408,48.85384


In [8]:
# definition of a function to check the unique values in each column

def unique_counts_col(df):
    for col in df.columns:
        print(f'Column: {col}')
        print(df[col].value_counts())
        print('-----------------------------')

unique_counts_col(airbnb)

Column: realSum
realSum
243.265915     42
278.217914     40
289.868580     38
266.567248     38
301.286234     34
               ..
257.479728      1
4188.414577     1
138.875944      1
142.837170      1
181.517383      1
Name: count, Length: 1365, dtype: int64
-----------------------------
Column: room_type
room_type
Entire home/apt    2742
Private room        769
Shared room          47
Name: count, dtype: int64
-----------------------------
Column: room_shared
room_shared
False    3511
True       47
Name: count, dtype: int64
-----------------------------
Column: room_private
room_private
False    2789
True      769
Name: count, dtype: int64
-----------------------------
Column: person_capacity
person_capacity
2.0    1940
4.0     953
3.0     344
6.0     200
5.0     121
Name: count, dtype: int64
-----------------------------
Column: host_is_superhost
host_is_superhost
False    3046
True      512
Name: count, dtype: int64
-----------------------------
Column: cleanliness_rating
cleanli

In [9]:
# dropping more columns after checking the unique values
airbnb.drop(columns=['room_shared', 'room_private'], inplace=True)

In [10]:
# renaming columns for better understanding
airbnb.rename(columns={'realSum': 'cost_2pax_2night', 'lat': 'latitude', 'lng': 'longitude', 'dist': 'distance_km'}, inplace=True)

In [11]:
# checking further dropping and renaming
airbnb.head()

,cost_2pax_2night,room_type,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,bedrooms,distance_km,longitude,latitude
0,536.396682,Entire home/apt,5.0,False,9.0,89.0,1,1.351201,2.35900,48.86800
1,290.101594,Private room,2.0,True,10.0,97.0,1,0.699821,2.35385,48.86282
2,445.754497,Entire home/apt,4.0,False,10.0,100.0,1,0.968982,2.36023,48.86375
3,211.343089,Private room,2.0,False,10.0,94.0,1,3.302319,2.31714,48.87475
4,266.334234,Entire home/apt,2.0,True,9.0,88.0,1,1.402430,2.33408,48.85384


In [12]:
# creating a new column to differentiate between weekday and weekend listings (for when dfs are merged later on)
airbnb['type_of_day'] = 'weekend'


In [13]:
# creating a new column with the cost per day for 2 pax
airbnb['cost_2pax_1night'] = airbnb['cost_2pax_2night'] / 2

In [14]:
# checking column creation
airbnb.head()

,cost_2pax_2night,room_type,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,bedrooms,distance_km,longitude,latitude,type_of_day,cost_2pax_1night
0,536.396682,Entire home/apt,5.0,False,9.0,89.0,1,1.351201,2.35900,48.86800,weekend,268.198341
1,290.101594,Private room,2.0,True,10.0,97.0,1,0.699821,2.35385,48.86282,weekend,145.050797
2,445.754497,Entire home/apt,4.0,False,10.0,100.0,1,0.968982,2.36023,48.86375,weekend,222.877249
3,211.343089,Private room,2.0,False,10.0,94.0,1,3.302319,2.31714,48.87475,weekend,105.671544
4,266.334234,Entire home/apt,2.0,True,9.0,88.0,1,1.402430,2.33408,48.85384,weekend,133.167117


In [15]:
# reordering columns for better understanding
airbnb = airbnb[['type_of_day', 'cost_2pax_2night', 'cost_2pax_1night', 'room_type', 'bedrooms', 'person_capacity', 'host_is_superhost', 'cleanliness_rating', 'guest_satisfaction_overall', 'longitude', 'latitude', 'distance_km']]

In [16]:
# checking the final dataframe
airbnb.head()

,type_of_day,cost_2pax_2night,cost_2pax_1night,room_type,bedrooms,person_capacity,host_is_superhost,cleanliness_rating,guest_satisfaction_overall,longitude,latitude,distance_km
0,weekend,536.396682,268.198341,Entire home/apt,1,5.0,False,9.0,89.0,2.35900,48.86800,1.351201
1,weekend,290.101594,145.050797,Private room,1,2.0,True,10.0,97.0,2.35385,48.86282,0.699821
2,weekend,445.754497,222.877249,Entire home/apt,1,4.0,False,10.0,100.0,2.36023,48.86375,0.968982
3,weekend,211.343089,105.671544,Private room,1,2.0,False,10.0,94.0,2.31714,48.87475,3.302319
4,weekend,266.334234,133.167117,Entire home/apt,1,2.0,True,9.0,88.0,2.33408,48.85384,1.402430


In [17]:
# exporting the cleaned dataframe
airbnb.to_pickle('../01. Data/02. Clean Data/airbnb_paris_weekend_cleaned.pkl')